# BBC Full Text Document Classification Dataset

## 1. Dataset Description

The **BBC Full Text Document Classification Dataset** is a collection of news articles designed for text classification tasks using Natural Language Processing (NLP) and Machine Learning techniques.

The dataset contains **2,225 full-text news articles** collected from BBC News during 2004–2005. Each article is assigned to one of five predefined categories, making it a multi-class text classification dataset.

### Dataset Categories

The dataset is divided into five categories:

- **Business:** Articles related to business, companies, markets, and the economy.
- **Entertainment:** Articles covering movies, music, celebrities, and cultural events.
- **Politics:** Articles about political events, governments, and public affairs.
- **Sport:** Articles covering sports events, teams, and athletes.
- **Tech:** Articles related to technology, computing, and digital developments.

### Dataset Statistics

| Property | Description |
|---|---|
| Dataset Name | BBC Full Text Document Classification |
| Total Documents | 2,225 |
| Number of Classes | 5 |
| Language | English |
| Data Type | Text |
| File Format | TXT |
| Task | Multi-class Text Classification |
| Source | BBC News (2004–2005) |

### Project Objective

The main objective of this project is to develop an **AI-powered News Classification and Intelligent Routing System** that automatically analyzes news articles, predicts their categories, and routes them to the corresponding departments.

The project will explore and compare different approaches, including traditional Machine Learning algorithms, Deep Learning models, and Transformer-based models.

### Classification and Routing

Each predicted category will be mapped to its corresponding department:

| Predicted Category | Assigned Department |
|---|---|
| Business | Business Department |
| Entertainment | Entertainment Department |
| Politics | Political Affairs Department |
| Sport | Sports Department |
| Tech | Technology Department |

## Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import re

import nltk

from nltk.corpus import stopwords


In [ ]:
data=pd.read_csv("../data/raw/bbc_data.csv")
data.info()

In [ ]:
data.describe(include='all')

In [ ]:
data.isnull().sum()

In [ ]:
data.shape

In [ ]:
data.duplicated().sum()

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
plt.figure(figsize=(10, 5))

sns.countplot( data , x='labels' , hue='labels')

plt.title('Distribution of News Categories')
plt.xlabel('News Category')
plt.ylabel('Number of text')
plt.show()

In [ ]:
def word_count (data_text):
    return len(data_text.split())

data['word_count'] = data['data'].apply(word_count)
data.head()

In [ ]:
data['word_count'].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(data=data, x='word_count', bins=30, kde=True)

plt.title('Distribution of Article Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=data,
    x='labels',
    y='word_count',
    hue='labels'
    )

plt.title('Average Article Length per Category')
plt.xlabel('News Category')
plt.ylabel('Average Number of Words')
plt.show()

## Text Preprocessing 
- Converted text to lowercase.
- Removed punctuation and unnecessary characters.
- Removed standalone numbers.
- Tokenized articles using NLTK.
- Removed English stopwords while preserving negation words.
- Applied Lemmatization using WordNet and POS Tagging.
- Reconstructed processed tokens into clean text.

In [ ]:
clean_data = data.copy()

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

clean_data['data'] = clean_data['data'].apply(clean_text)

In [ ]:
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

In [ ]:
def tokenize_text(text):
    tokens = word_tokenize(text)
    return tokens


clean_data['tokens'] = clean_data['data'].apply(tokenize_text)

In [ ]:
len(clean_data['tokens'].iloc[0])

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words.difference_update({'no', 'nor', 'not'})

In [ ]:
def remove_stopwords(tokens):
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return filtered_tokens

clean_data['tokens'] = clean_data['tokens'].apply(remove_stopwords)

In [ ]:
len(clean_data['tokens'].iloc[0])

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
def lemmatize_text(tokens):
    
    tagged_words = pos_tag(tokens)
    lemmatized_tokens = []

    for word, tag in tagged_words:
        if tag.startswith('J'):
            pos = wordnet.ADJ
        elif tag.startswith('V'):
            pos = wordnet.VERB
        elif tag.startswith('N'):
            pos = wordnet.NOUN
        elif tag.startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN

        lemmatized_tokens.append(lemmatizer.lemmatize(word, pos))

    return lemmatized_tokens

clean_data['tokens'] = clean_data['tokens'].apply(lemmatize_text)

In [ ]:
clean_data['processed_text'] = clean_data['tokens'].apply(lambda tokens: ' '.join(tokens))

In [ ]:
clean_data[['data', 'tokens', 'processed_text']].head()

### TF-IDF (Term Frequency–Inverse Document Frequency)

In [ ]:
clean_data.info()

In [ ]:
X = clean_data['processed_text']
y = clean_data['labels']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
first = X_train_tfidf[0]

print(first)

### Machine Learning Model (Logistic Regression)

In [ ]:

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = logistic_model.predict(X_test_tfidf)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

In [ ]:
report = classification_report(y_test, y_pred)

print(report)


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12,8))
sns.heatmap(cm , annot=True , cmap='Reds')
plt.title('Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

plt.show()

### Machine Learning Model (Linear SVC)

In [ ]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=2000
)

svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_test_tfidf)

In [ ]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", accuracy_svm)
print("SVM Accuracy Percentage:", accuracy_svm * 100, "%")

print("\nSVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

cm_svm = confusion_matrix(y_test, y_pred_svm)



In [ ]:
cm_svm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(12,8))
sns.heatmap(cm_svm, annot=True , cmap='Reds')
plt.title('Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

plt.show()

In [ ]:
import joblib
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

# Define Models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "SVM": LinearSVC(
        C=1.0,
        random_state=42,
        max_iter=2000
    )
}

results = {}
pipelines = {}

for name, model in models.items():

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2)
        )),
        ('classifier', model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro')

    results[name] = {
        'Accuracy': accuracy,
        'F1-Score': f1
    }

    pipelines[name] = pipeline

    print(f"\n{name}")
    print("Accuracy:", accuracy)
    print("F1-Score:", f1)

best_model_name = max(
    results,
    key=lambda name: results[name]['F1-Score']
)

best_pipeline = pipelines[best_model_name]

print("\nBest Model:", best_model_name)
print("Best Results:", results[best_model_name])

joblib.dump(best_pipeline, '../models/best_news_classifier.pkl')

print("\nBest Pipeline Saved Successfully!")